In [ ]:
import torch
from torch import nn

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
lemmatizer = WordNetLemmatizer()

In [ ]:
import kagglehub
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
fake = pd.read_csv(path + "/Fake.csv")
real = pd.read_csv(path + "/True.csv")
fake['label'] = 1
real['label'] = 0

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, output_dim=2):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        x = self.dropout(hidden[-1])
        return self.fc(x)

In [ ]:
checkpoint = torch.load('lstm_fake_news_model.pth', map_location=torch.device('cpu'))
vocab = checkpoint['vocab']
params = checkpoint['params']

In [ ]:
model = LSTMClassifier(
    vocab_size=len(vocab),
    embed_dim=params['embed_dim'],
    hidden_dim=params['hidden_dim'],
    dropout=params['dropout']
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

LSTMClassifier(
  (embedding): Embedding(20002, 256, padding_idx=0)
  (lstm): LSTM(256, 128, batch_first=True)
  (dropout): Dropout(p=0.3585718875304013, inplace=False)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)

In [ ]:
class FakeNewsPredictor:
    def __init__(self, model, vocab, max_len=500):
        self.model = model
        self.vocab = vocab
        self.max_len = max_len

    def preprocess(self, text):
        # copy your preprocess_text() here
        text = text.lower()
        text = re.sub(r'[^a-z\s]', '', text)
        tokens = text.split()
        tokens = [word for word in tokens if word not in stop_words]
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        return ' '.join(tokens)

    def encode(self, text):
        tokens = text.split()
        encoded = [self.vocab.get(word, self.vocab['<UNK>']) for word in tokens[:self.max_len]]
        return encoded + [self.vocab['<PAD>']] * (self.max_len - len(encoded))

    def predict(self, text):
        self.model.eval()
        cleaned = self.preprocess(text)
        encoded = self.encode(cleaned)
        tensor = torch.tensor(encoded, dtype=torch.long).unsqueeze(0)
        with torch.no_grad():
            output = self.model(tensor)
            pred = torch.argmax(output, dim=1).item()
        return "Fake" if pred == 1 else "Real"


In [ ]:
predictor = FakeNewsPredictor(model, vocab)
sample = real.iloc[100]['title'] + ' ' + real.iloc[100]['text']
print(type(sample))
result = predictor.predict(sample)
print(result)

<class 'str'>
Real
